# Tool Call Error Analysis (SQL)
Uses DuckDB for SQL-based querying of inference results. Each section has a query you can tweak to explore examples.

In [ ]:
import json, duckdb

DATA_PATH = "../output/inference/pd_fin_llm_financial_tool_calling.jsonl"

with open(DATA_PATH) as f:
    samples = [json.loads(line) for line in f]

def parse_tool(text):
    try:
        d = json.loads(text)
        if isinstance(d, dict) and "name" in d:
            return d
    except Exception:
        pass
    return None

rows = []
for s in samples:
    ptc = parse_tool(s["prediction"])
    rtc = parse_tool(s["reference"])
    pred_params = json.dumps(ptc.get("parameters", {}), sort_keys=True) if ptc else None
    ref_params = json.dumps(rtc.get("parameters", {}), sort_keys=True) if rtc else None

    # per-key diffs
    missing_keys = extra_keys = wrong_keys = None
    if ptc and rtc:
        pp, rp = ptc.get("parameters", {}), rtc.get("parameters", {})
        missing_keys = sorted(set(rp) - set(pp)) or None
        extra_keys = sorted(set(pp) - set(rp)) or None
        wrong_keys = sorted(k for k in set(pp) & set(rp) if pp[k] != rp[k]) or None

    rows.append({
        "idx": s["index"],
        "question": s["question"],
        "pred_raw": s["prediction"],
        "ref_raw": s["reference"],
        "pred_is_tool": ptc is not None,
        "ref_is_tool": rtc is not None,
        "pred_name": ptc["name"] if ptc else None,
        "ref_name": rtc["name"] if rtc else None,
        "pred_params": pred_params,
        "ref_params": ref_params,
        "missing_keys": json.dumps(missing_keys) if missing_keys else None,
        "extra_keys": json.dumps(extra_keys) if extra_keys else None,
        "wrong_value_keys": json.dumps(wrong_keys) if wrong_keys else None,
    })

import pandas as pd
con = duckdb.connect()
df_rows = pd.DataFrame(rows)
con.execute("CREATE TABLE results AS SELECT * FROM df_rows")
con.sql("SELECT count(*) AS total FROM results").show()

## Helper to run queries

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

def q(sql):
    return con.sql(sql).df()

## 1. Call-Type Confusion Matrix

In [ ]:
q("""
SELECT
    CASE WHEN ref_is_tool THEN 'tool' ELSE 'text' END AS reference,
    CASE WHEN pred_is_tool THEN 'tool' ELSE 'text' END AS prediction,
    count(*) AS cnt
FROM results
GROUP BY 1, 2
ORDER BY 1, 2
""")

## 2. Missed Tool Calls (should call tool, gave text)

In [ ]:
q("""
SELECT idx, question, ref_raw AS expected, pred_raw AS got
FROM results
WHERE ref_is_tool AND NOT pred_is_tool
""")

## 3. Hallucinated Tool Calls (should give text, called tool)

In [ ]:
q("""
SELECT idx, question, pred_name, pred_params
FROM results
WHERE NOT ref_is_tool AND pred_is_tool
""")

## 4. Wrong Tool Name

In [ ]:
q("""
SELECT idx, question, ref_name AS expected_tool, pred_name AS predicted_tool
FROM results
WHERE pred_is_tool AND ref_is_tool AND pred_name != ref_name
""")

In [ ]:
q("""
SELECT ref_name || ' -> ' || pred_name AS confusion_pair, count(*) AS cnt
FROM results
WHERE pred_is_tool AND ref_is_tool AND pred_name != ref_name
GROUP BY 1 ORDER BY 2 DESC
""")

## 5. Wrong Parameters (correct tool name)

In [ ]:
q("""
SELECT
    count(*) AS total_correct_name,
    count(*) FILTER (WHERE pred_params = ref_params) AS exact_param_match,
    count(*) FILTER (WHERE pred_params != ref_params) AS param_mismatch
FROM results
WHERE pred_is_tool AND ref_is_tool AND pred_name = ref_name
""")

In [ ]:
q("""
SELECT idx, question, ref_name AS tool,
       missing_keys, extra_keys, wrong_value_keys,
       ref_params, pred_params
FROM results
WHERE pred_is_tool AND ref_is_tool AND pred_name = ref_name
  AND pred_params != ref_params
ORDER BY idx
""")

### Most common parameter errors

In [ ]:
q("""
WITH errs AS (
    SELECT unnest(from_json(missing_keys, '"json"')::varchar[]) AS key, 'missing' AS error_type
    FROM results WHERE missing_keys IS NOT NULL
    UNION ALL
    SELECT unnest(from_json(extra_keys, '"json"')::varchar[]), 'extra'
    FROM results WHERE extra_keys IS NOT NULL
    UNION ALL
    SELECT unnest(from_json(wrong_value_keys, '"json"')::varchar[]), 'wrong_value'
    FROM results WHERE wrong_value_keys IS NOT NULL
)
SELECT error_type, key, count(*) AS cnt
FROM errs GROUP BY 1, 2 ORDER BY 3 DESC
""")

## 6. Summary

In [ ]:
q("""
SELECT
    count(*) AS total,
    count(*) FILTER (WHERE NOT ref_is_tool AND NOT pred_is_tool) AS both_text,
    count(*) FILTER (WHERE ref_is_tool AND NOT pred_is_tool) AS missed_call,
    count(*) FILTER (WHERE NOT ref_is_tool AND pred_is_tool) AS hallucinated_call,
    count(*) FILTER (WHERE pred_is_tool AND ref_is_tool AND pred_name != ref_name) AS wrong_name,
    count(*) FILTER (WHERE pred_is_tool AND ref_is_tool AND pred_name = ref_name AND pred_params != ref_params) AS wrong_params,
    count(*) FILTER (WHERE pred_is_tool AND ref_is_tool AND pred_name = ref_name AND pred_params = ref_params) AS perfect_tool_call
FROM results
""")

In [ ]:
import matplotlib.pyplot as plt

s = q("""
SELECT * FROM (
    VALUES
        ('Perfect tool call',  count(*) FILTER (WHERE pred_is_tool AND ref_is_tool AND pred_name = ref_name AND pred_params = ref_params)),
        ('Wrong params',       count(*) FILTER (WHERE pred_is_tool AND ref_is_tool AND pred_name = ref_name AND pred_params != ref_params)),
        ('Wrong tool name',    count(*) FILTER (WHERE pred_is_tool AND ref_is_tool AND pred_name != ref_name)),
        ('Missed call',        count(*) FILTER (WHERE ref_is_tool AND NOT pred_is_tool)),
        ('Hallucinated call',  count(*) FILTER (WHERE NOT ref_is_tool AND pred_is_tool)),
        ('Both text (correct)',count(*) FILTER (WHERE NOT ref_is_tool AND NOT pred_is_tool))
) t(category, cnt)
FROM results
""")

colors = ["#4CAF50", "#FF9800", "#F44336", "#9C27B0", "#E91E63", "#8BC34A"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(s.category, s.cnt, color=colors)
for i, v in enumerate(s.cnt):
    ax.text(v + 0.5, i, str(v), va="center")
ax.set_xlabel("Count")
ax.set_title("Tool Call Error Breakdown")
plt.tight_layout()
plt.show()

## 7. Ad-hoc Exploration
Edit the SQL below to drill into any subset you want.

In [ ]:
# Example: show all samples where 'time_range' param was wrong
q("""
SELECT idx, question, ref_name AS tool, ref_params, pred_params
FROM results
WHERE wrong_value_keys LIKE '%time_range%'
""")

In [ ]:
# Example: show full prediction/reference for a specific sample
q("""
SELECT idx, question, pred_raw, ref_raw
FROM results
WHERE idx = 3
""")